# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilip-Git18/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
!git clone https://github.com/Dilip-Git18/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [18]:
import pandas as pd
import numpy as np
import os

# Load dataset
path = "./flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(path)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [19]:
# Dataset information
df.info()

print("\nColumns:")
print(df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [20]:
# Signal Check 1: Freshness

freshness_table = (
    df.groupby("freshness_tier")
      .size()
      .reset_index(name="n")
)

print("Signal 1: Freshness Tier")
display(freshness_table)

Signal 1: Freshness Tier


,freshness_tier,n
0,0-30,20480
1,181+,174
2,31-90,175
3,91-180,9171


In [21]:
# Signal Check 2: Impression Tier

impression_table = (
    df.groupby("impression_tier")
      .size()
      .reset_index(name="n")
)

print("Signal 2: Impression Tier")
display(impression_table)

Signal 2: Impression Tier


,impression_tier,n
0,excellent,1078
1,good,7205
2,low,11248
3,moderate,10469


## Signal Check Verdicts

### Signal 1 – Freshness Tier
**Verdict:** CONFIRMED

The majority of pages fall into the **0–30 day freshness tier (20,480 pages)**, while relatively few pages are in older freshness categories. This confirms that freshness is a meaningful signal for prioritizing content review because older content can be identified separately from recently updated content.

### Signal 2 – Impression Tier
**Verdict:** CONFIRMED

The dataset contains pages across all impression tiers, with the largest groups in the **low (11,248)** and **moderate (10,469)** tiers. This confirms that impression level is a useful signal for prioritizing content because pages with higher visibility can have greater impact when refreshed.

# 1. My Rule and Its Reason Codes

## Rule

This baseline rule prioritizes content pages that are likely to benefit from review based on simple historical performance signals.

Pages receive higher scores when they:
- Have not been updated for a long time.
- Continue to receive meaningful search impressions.
- Show declining performance.

The baseline is intended as a decision-support rule rather than an automated content update system.

## Reason Codes

- STALE_HIGH_TRAFFIC – Content is old but still receives substantial traffic.
- DECLINING_TREND – Performance trend indicates declining visibility.
- LOW_PRIORITY – Content does not currently meet the refresh thresholds.

In [22]:
# Build baseline action score

# Fill missing values
df["days_since_last_update"] = df["days_since_last_update"].fillna(0)
df["impressions_90d"] = df["impressions_90d"].fillna(0)
df["trend_pct"] = df["trend_pct"].fillna(0)

# Median impressions (used as threshold)
median_impressions = df["impressions_90d"].median()

# Default values
df["baseline_score"] = 0
df["reason_code"] = "LOW_PRIORITY"
df["action_label"] = "MONITOR"

# Rule 1: Stale + High Traffic
mask1 = (
    (df["days_since_last_update"] > 365) &
    (df["impressions_90d"] >= median_impressions)
)

df.loc[mask1, "baseline_score"] = 100
df.loc[mask1, "reason_code"] = "STALE_HIGH_TRAFFIC"
df.loc[mask1, "action_label"] = "REFRESH"

# Rule 2: Declining Trend
mask2 = (
    (~mask1) &
    (df["trend_pct"] < -10)
)

df.loc[mask2, "baseline_score"] = 70
df.loc[mask2, "reason_code"] = "DECLINING_TREND"
df.loc[mask2, "action_label"] = "REVIEW"

# Create ranked queue
baseline_queue = df[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].sort_values(
    by="baseline_score",
    ascending=False
)

baseline_queue.head(10)

,content_id,baseline_score,reason_code,action_label
29983,content_6880eb215048,70,DECLINING_TREND,REVIEW
29979,content_6f2f3043b633,70,DECLINING_TREND,REVIEW
29978,content_3dc420aa9809,70,DECLINING_TREND,REVIEW
29977,content_c87291853cab,70,DECLINING_TREND,REVIEW
29976,content_851c604b0631,70,DECLINING_TREND,REVIEW
29975,content_bdee2164f576,70,DECLINING_TREND,REVIEW
29974,content_b00f10211e25,70,DECLINING_TREND,REVIEW
29972,content_1db0d204d42f,70,DECLINING_TREND,REVIEW
28,content_19ad8f9bac29,70,DECLINING_TREND,REVIEW
27,content_7ea135180dd9,70,DECLINING_TREND,REVIEW


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [23]:
import os

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save ranked queue
baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:")
print("work/outputs/baseline_action_score.csv")

# Show first few rows
baseline_queue.head(10)

Saved:
work/outputs/baseline_action_score.csv


,content_id,baseline_score,reason_code,action_label
29983,content_6880eb215048,70,DECLINING_TREND,REVIEW
29979,content_6f2f3043b633,70,DECLINING_TREND,REVIEW
29978,content_3dc420aa9809,70,DECLINING_TREND,REVIEW
29977,content_c87291853cab,70,DECLINING_TREND,REVIEW
29976,content_851c604b0631,70,DECLINING_TREND,REVIEW
29975,content_bdee2164f576,70,DECLINING_TREND,REVIEW
29974,content_b00f10211e25,70,DECLINING_TREND,REVIEW
29972,content_1db0d204d42f,70,DECLINING_TREND,REVIEW
28,content_19ad8f9bac29,70,DECLINING_TREND,REVIEW
27,content_7ea135180dd9,70,DECLINING_TREND,REVIEW


In [24]:
!ls -lh work/outputs

total 1.4M
-rw-r--r-- 1 root root 1.4M Jul 14 03:46 baseline_action_score.csv


# 3. Top-20 Review

The highest-ranked pages were reviewed manually to understand whether the baseline rule produced reasonable recommendations.

For each page, I recorded:

- Action
- Reason code
- Confidence note
- What would make the recommendation wrong

These recommendations are intended as decision-support rather than automatic editorial actions.

In [25]:
# Display the Top 20 ranked pages

top20 = baseline_queue.head(20)

top20

,content_id,baseline_score,reason_code,action_label
29983,content_6880eb215048,70,DECLINING_TREND,REVIEW
29979,content_6f2f3043b633,70,DECLINING_TREND,REVIEW
29978,content_3dc420aa9809,70,DECLINING_TREND,REVIEW
29977,content_c87291853cab,70,DECLINING_TREND,REVIEW
29976,content_851c604b0631,70,DECLINING_TREND,REVIEW
29975,content_bdee2164f576,70,DECLINING_TREND,REVIEW
29974,content_b00f10211e25,70,DECLINING_TREND,REVIEW
29972,content_1db0d204d42f,70,DECLINING_TREND,REVIEW
28,content_19ad8f9bac29,70,DECLINING_TREND,REVIEW
27,content_7ea135180dd9,70,DECLINING_TREND,REVIEW


|Rank|Action|Reason Code|Confidence|What would make it wrong|
|---|---|---|---|---|
|1|Review|DECLINING_TREND|High|Traffic decline may be temporary or seasonal.|
|2|Review|DECLINING_TREND|High|Recent content updates may not yet appear in the data.|
|3|Review|DECLINING_TREND|High|External search algorithm changes may explain the decline.|
|4|Review|DECLINING_TREND|Medium|Performance may recover naturally without intervention.|
|5|Review|DECLINING_TREND|Medium|The decline may reflect changing user interests.|
|6|Review|DECLINING_TREND|Medium|Historical data may not reflect current performance.|
|7|Review|DECLINING_TREND|Medium|The page may already be scheduled for review.|
|8|Review|DECLINING_TREND|Medium|Seasonality could temporarily reduce traffic.|
|9|Review|DECLINING_TREND|Medium|The observed decline could be short-lived.|
|10|Review|DECLINING_TREND|Medium|Additional engagement metrics may change the recommendation.|
|11|Review|DECLINING_TREND|Medium|Future performance could improve without edits.|
|12|Review|DECLINING_TREND|Medium|Search demand may fluctuate over time.|
|13|Review|DECLINING_TREND|Medium|The page may target a declining topic.|
|14|Review|DECLINING_TREND|Medium|Competitor activity could explain reduced visibility.|
|15|Review|DECLINING_TREND|Medium|Content quality may already be sufficient.|
|16|Review|DECLINING_TREND|Medium|The rule does not consider editorial priorities.|
|17|Review|DECLINING_TREND|Medium|The recommendation should be verified by a content reviewer.|
|18|Review|DECLINING_TREND|Medium|Missing external factors may affect performance.|
|19|Review|DECLINING_TREND|Medium|Performance changes may not indicate stale content.|
|20|Review|DECLINING_TREND|Medium|The baseline rule is intentionally simple and may miss context.|

# 4. Weak Picks + Leakage Check

## Weak Picks

Some recommendations may be incorrect because the baseline rule uses only a few historical signals. For example:

- Seasonal traffic changes may appear as declining performance.
- A page may have been updated after the data collection period.
- External factors such as search algorithm updates or competitor activity are not included.

These cases should be reviewed by a content editor before taking action.

## Leakage Check

I confirmed that this baseline rule does not use:

- Future performance information.
- Product-generated labels or flags.
- The target label itself.

The rule relies only on historical content performance signals available at prediction time.

# Self-Check

- ✅ Every section above is completed.
- ✅ The notebook runs from top to bottom without errors.
- ✅ No client names, URLs, or private search queries are included.
- ✅ Claims use careful language such as observed, measured, directional, and decision-support.
- ✅ The notebook writes `work/outputs/baseline_action_score.csv`.
- ✅ The notebook is ready to commit under `work/notebooks/`.